In [16]:
# 1. 라이브러리

import os
import copy
import numpy as np
import pandas as pd
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler

import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

In [17]:
import torch
print(torch.cuda.get_device_name(0))

NVIDIA GeForce RTX 4060


In [18]:
# 2. 4~8월 cargo CSV 로딩

file_paths = [
    r"C:\Users\HP\Desktop\prE\04cargo.csv",
    r"C:\Users\HP\Desktop\prE\05cargo.csv",
    r"C:\Users\HP\Desktop\prE\06cargo.csv",
    r"C:\Users\HP\Desktop\prE\07cargo.csv",
    r"C:\Users\HP\Desktop\prE\08cargo.csv",
]

df_list = []

for path in file_paths:
    temp = pd.read_csv(path)
    temp["source_file"] = os.path.basename(path)
    df_list.append(temp)

df_raw = pd.concat(df_list, ignore_index=True)

print(df_raw.shape)
print(df_raw.columns)
df_raw.head()

(4500839, 20)
Index(['MMSI', 'Date', 'Lat', 'Long', 'SOG', 'COG', 'Heading', 'Name', 'Type',
       'IMO', 'Call', 'DimA', 'DimB', 'DimC', 'DimD', 'Draft', 'Tons',
       'Length', 'Breadth', 'source_file'],
      dtype='str')


,MMSI,Date,Lat,Long,SOG,COG,Heading,Name,Type,IMO,Call,DimA,DimB,DimC,DimD,Draft,Tons,Length,Breadth,source_file
0,215331000,2024-04-01 00:02:56,35.067482,128.808068,0.0,285.7,78,CMA CGM ARGENTINA,70.0,9839909.0,9HA5066,110.0,256.0,29.0,22.0,14.4,169408.0,366.0,51.0,04cargo.csv
1,215331000,2024-04-01 00:05:56,35.067487,128.808107,0.0,233.2,78,CMA CGM ARGENTINA,70.0,9839909.0,9HA5066,110.0,256.0,29.0,22.0,14.4,169408.0,366.0,51.0,04cargo.csv
2,215331000,2024-04-01 00:08:56,35.067478,128.808130,0.0,236.5,78,CMA CGM ARGENTINA,70.0,9839909.0,9HA5066,110.0,256.0,29.0,22.0,14.4,169408.0,366.0,51.0,04cargo.csv
3,215331000,2024-04-01 00:11:56,35.067490,128.808120,0.0,242.4,78,CMA CGM ARGENTINA,70.0,9839909.0,9HA5066,110.0,256.0,29.0,22.0,14.4,169408.0,366.0,51.0,04cargo.csv
4,215331000,2024-04-01 00:14:56,35.067518,128.808348,0.0,238.9,78,CMA CGM ARGENTINA,70.0,9839909.0,9HA5066,110.0,256.0,29.0,22.0,14.4,169408.0,366.0,51.0,04cargo.csv


In [19]:
# 3. 기본 전처리

df = df_raw[["MMSI", "Date", "Lat", "Long", "SOG", "COG", "Heading", "source_file"]].copy()

df = df.rename(columns={
    "Date": "timestamp",
    "Lat": "lat",
    "Long": "lon",
    "SOG": "sog",
    "COG": "cog",
    "Heading": "heading"
})

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

for col in ["lat", "lon", "sog", "cog", "heading"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["MMSI", "timestamp", "lat", "lon", "sog", "cog", "heading"])

df = df[
    (df["lat"].between(-90, 90)) &
    (df["lon"].between(-180, 180)) &
    (df["sog"] >= 0) &
    (df["cog"].between(0, 360)) &
    (df["heading"].between(0, 360))
].copy()

df = df.sort_values(["MMSI", "timestamp"]).reset_index(drop=True)
df["month"] = df["timestamp"].dt.month

print(df.shape)
print(df["month"].value_counts().sort_index())
df.head()

(4417967, 9)
month
4    826933
5    900471
6    907346
7    912136
8    871081
Name: count, dtype: int64


,MMSI,timestamp,lat,lon,sog,cog,heading,source_file,month
0,209079000,2024-08-05 03:42:22,34.960100,128.823167,9.9,339.7,338,08cargo.csv,8
1,209079000,2024-08-05 03:42:32,34.960518,128.822972,9.8,339.7,338,08cargo.csv,8
2,209079000,2024-08-05 03:42:42,34.960935,128.822775,9.6,339.1,338,08cargo.csv,8
3,209079000,2024-08-05 03:42:52,34.961335,128.822592,9.4,338.6,338,08cargo.csv,8
4,209079000,2024-08-05 03:43:02,34.961732,128.822387,9.3,338.0,338,08cargo.csv,8


In [20]:
# 4. 선박 상태 라벨 생성
# 학습/검증/테스트에는 전체 AIS 데이터를 사용한다.
# 단, sog 기준으로 이동/저속 상태를 라벨링해서 뒤쪽 분석에 활용한다.

MOVING_THRESHOLD = 0.5

moving_df = df.copy()
moving_df["is_moving"] = moving_df["sog"] > MOVING_THRESHOLD

stationary_df = moving_df[moving_df["sog"] <= MOVING_THRESHOLD].copy()
active_df = moving_df[moving_df["sog"] > MOVING_THRESHOLD].copy()

print("전체 데이터:", len(df))
print("학습/예측 사용 데이터:", len(moving_df))
print("이동 상태 데이터:", len(active_df))
print("정지/저속 상태 데이터:", len(stationary_df))
print("이동 상태 비율:", round(len(active_df) / len(moving_df) * 100, 2), "%")
print("정지/저속 상태 비율:", round(len(stationary_df) / len(moving_df) * 100, 2), "%")

전체 데이터: 4417967
학습/예측 사용 데이터: 4417967
이동 상태 데이터: 2614784
정지/저속 상태 데이터: 1803183
이동 상태 비율: 59.19 %
정지/저속 상태 비율: 40.81 %


In [21]:
# 원본 moving_df 기준 AIS 수신 간격 분포 확인

interval_df = []

for mmsi, group in moving_df.groupby("MMSI"):
    group = group.sort_values("timestamp")
    dt_sec = group["timestamp"].diff().dt.total_seconds().dropna()

    if len(dt_sec) == 0:
        continue

    interval_df.append(pd.DataFrame({
        "MMSI": mmsi,
        "dt_sec": dt_sec
    }))

interval_df = pd.concat(interval_df, ignore_index=True)

display(interval_df["dt_sec"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

print("30초 이하 비율:", (interval_df["dt_sec"] <= 30).mean())
print("60초 이하 비율:", (interval_df["dt_sec"] <= 60).mean())
print("120초 이하 비율:", (interval_df["dt_sec"] <= 120).mean())
print("300초 이하 비율:", (interval_df["dt_sec"] <= 300).mean())

count    4.416601e+06
mean     1.242474e+03
std      7.518224e+04
min      0.000000e+00
10%      5.000000e+00
25%      9.000000e+00
50%      1.000000e+01
75%      1.200000e+01
90%      1.800000e+02
max      1.249734e+07
Name: dt_sec, dtype: float64

30초 이하 비율: 0.7840932427448166
60초 이하 비율: 0.7849656330739408
120초 이하 비율: 0.786191236201776
300초 이하 비율: 0.9926194374361641


In [22]:
# 5. Grid 설정

TRAIN_GRID_NM = 0.1
BASE_LAT_FOR_LON_SCALE = 35.0

TRAIN_LAT_GRID_SIZE = (TRAIN_GRID_NM * 1.852) / 111
TRAIN_LON_GRID_SIZE = (TRAIN_GRID_NM * 1.852) / (
    111 * np.cos(np.radians(BASE_LAT_FOR_LON_SCALE))
)

print("TRAIN_LAT_GRID_SIZE:", TRAIN_LAT_GRID_SIZE)
print("TRAIN_LON_GRID_SIZE:", TRAIN_LON_GRID_SIZE)

TRAIN_LAT_GRID_SIZE: 0.0016684684684684687
TRAIN_LON_GRID_SIZE: 0.0020368239084560514


In [23]:
# 6. 30초 단위 resampling
# 선형보간 없음. 실제 관측이 있는 30초 bin만 사용.

RESAMPLE_INTERVAL = "30s"
STEP_SECONDS = 30

MAX_REASONABLE_SPEED_KNOTS = 40


def circular_mean_deg(series):
    values = series.dropna().values

    if len(values) == 0:
        return np.nan

    radians = np.radians(values)
    sin_mean = np.mean(np.sin(radians))
    cos_mean = np.mean(np.cos(radians))

    angle = np.degrees(np.arctan2(sin_mean, cos_mean))
    return angle + 360 if angle < 0 else angle


def haversine_km_np(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return R * 2 * np.arcsin(np.sqrt(a))


def resample_ais_30s(df, progress_step=10):
    result = []
    grouped = list(df.groupby("MMSI"))
    total = len(grouped)
    next_progress = progress_step

    print(f"30s resampling start: {total} vessels")

    for idx, (mmsi, group) in enumerate(grouped, start=1):
        group = group.sort_values("timestamp").set_index("timestamp")

        resampled = group[["lat", "lon", "sog"]].resample(RESAMPLE_INTERVAL).mean()
        resampled["cog"] = group["cog"].resample(RESAMPLE_INTERVAL).apply(circular_mean_deg)
        resampled["heading"] = group["heading"].resample(RESAMPLE_INTERVAL).apply(circular_mean_deg)

        resampled = resampled.dropna()
        resampled["MMSI"] = mmsi
        result.append(resampled.reset_index())

        progress = idx / total * 100 if total else 100
        if progress >= next_progress or idx == total:
            print(f"resampling progress: {progress:.1f}% ({idx}/{total})")
            next_progress += progress_step

    if len(result) == 0:
        return pd.DataFrame(columns=["timestamp", "lat", "lon", "sog", "cog", "heading", "MMSI"])

    return pd.concat(result, ignore_index=True)


def remove_unrealistic_jumps(df, max_speed_knots=MAX_REASONABLE_SPEED_KNOTS):
    cleaned = []
    removed = 0

    for mmsi, group in df.groupby("MMSI"):
        group = group.sort_values("timestamp").copy()

        prev_lat = group["lat"].shift(1)
        prev_lon = group["lon"].shift(1)
        prev_time = group["timestamp"].shift(1)

        dt_hours = (group["timestamp"] - prev_time).dt.total_seconds() / 3600
        dist_km = haversine_km_np(prev_lat, prev_lon, group["lat"], group["lon"])
        speed_knots = (dist_km / 1.852) / dt_hours

        valid = prev_time.isna() | (
            (dt_hours > 0) &
            (speed_knots <= max_speed_knots)
        )

        removed += int((~valid).sum())
        cleaned.append(group.loc[valid])

    result = pd.concat(cleaned, ignore_index=True) if cleaned else df.iloc[0:0].copy()
    print("Removed unrealistic jump rows:", removed)
    return result


def add_direction_features(df):
    result = df.copy()

    result["cog_sin"] = np.sin(np.radians(result["cog"]))
    result["cog_cos"] = np.cos(np.radians(result["cog"]))
    result["heading_sin"] = np.sin(np.radians(result["heading"]))
    result["heading_cos"] = np.cos(np.radians(result["heading"]))

    return result

In [24]:
# 7. Train / Validation / Test split
# Train: 4월
# Validation: 5~7월
# Test: 8월

train_month_raw = moving_df[moving_df["month"].isin([4])].copy()
val_month_raw = moving_df[moving_df["month"].isin([5, 6, 7])].copy()
test_month_raw = moving_df[moving_df["month"].isin([8])].copy()

train_raw = add_direction_features(remove_unrealistic_jumps(resample_ais_30s(train_month_raw)))
val_raw = add_direction_features(remove_unrealistic_jumps(resample_ais_30s(val_month_raw)))
test_raw = add_direction_features(remove_unrealistic_jumps(resample_ais_30s(test_month_raw)))

for split_df in [train_raw, val_raw, test_raw]:
    split_df["month"] = split_df["timestamp"].dt.month

print("train_raw:", train_raw.shape)
print("val_raw:", val_raw.shape)
print("test_raw:", test_raw.shape)

30s resampling start: 537 vessels
resampling progress: 10.1% (54/537)
resampling progress: 20.1% (108/537)
resampling progress: 30.2% (162/537)
resampling progress: 40.0% (215/537)
resampling progress: 50.1% (269/537)
resampling progress: 60.1% (323/537)
resampling progress: 70.0% (376/537)
resampling progress: 80.1% (430/537)
resampling progress: 90.1% (484/537)
resampling progress: 100.0% (537/537)
Removed unrealistic jump rows: 1
30s resampling start: 1096 vessels
resampling progress: 10.0% (110/1096)
resampling progress: 20.1% (220/1096)
resampling progress: 30.0% (329/1096)
resampling progress: 40.1% (439/1096)
resampling progress: 50.0% (548/1096)
resampling progress: 60.0% (658/1096)
resampling progress: 70.1% (768/1096)
resampling progress: 80.0% (877/1096)
resampling progress: 90.1% (987/1096)
resampling progress: 100.0% (1096/1096)
Removed unrealistic jump rows: 658
30s resampling start: 521 vessels
resampling progress: 10.2% (53/521)
resampling progress: 20.2% (105/521)
resa

In [25]:
# 8. 정규화
# target은 다음 30초 delta_lat / delta_lon

feature_cols = [
    "lat", "lon", "sog",
    "cog_sin", "cog_cos",
    "heading_sin", "heading_cos"
]
target_cols = ["delta_lat", "delta_lon"]

x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()


def build_delta_target_frame(raw_df):
    rows = []
    expected_delta = pd.Timedelta(seconds=STEP_SECONDS)

    for _, group in raw_df.groupby("MMSI"):
        group = group.sort_values("timestamp").reset_index(drop=True)

        next_lat = group["lat"].shift(-1)
        next_lon = group["lon"].shift(-1)
        next_time = group["timestamp"].shift(-1)

        is_continuous = (next_time - group["timestamp"]) == expected_delta

        delta_df = pd.DataFrame({
            "delta_lat": next_lat - group["lat"],
            "delta_lon": next_lon - group["lon"]
        })

        rows.append(delta_df.loc[is_continuous].dropna())

    if len(rows) == 0:
        return pd.DataFrame(columns=target_cols)

    return pd.concat(rows, ignore_index=True)


train_delta_targets = build_delta_target_frame(train_raw)

x_scaler.fit(train_raw[feature_cols])
y_scaler.fit(train_delta_targets[target_cols])


def apply_scaling(raw_df):
    norm_df = raw_df.copy()
    norm_df[feature_cols] = x_scaler.transform(raw_df[feature_cols])
    return norm_df


train_norm = apply_scaling(train_raw)
val_norm = apply_scaling(val_raw)
test_norm = apply_scaling(test_raw)

print("train delta target rows:", len(train_delta_targets))

train delta target rows: 194154


In [26]:
# 9. Interaction transition statistics

def latlon_to_grid(lat, lon, lat_min, lon_min, lat_grid_size, lon_grid_size):
    row = int((lat - lat_min) / lat_grid_size)
    col = int((lon - lon_min) / lon_grid_size)
    return row, col


def build_interaction_counts(df):
    transition_counts = defaultdict(int)

    lat_min = df["lat"].min()
    lon_min = df["lon"].min()

    for _, group in df.groupby("MMSI"):
        group = group.sort_values("timestamp")
        coords = group[["lat", "lon"]].values

        for i in range(len(coords) - 1):
            g1 = latlon_to_grid(
                coords[i][0], coords[i][1],
                lat_min, lon_min,
                TRAIN_LAT_GRID_SIZE, TRAIN_LON_GRID_SIZE
            )
            g2 = latlon_to_grid(
                coords[i + 1][0], coords[i + 1][1],
                lat_min, lon_min,
                TRAIN_LAT_GRID_SIZE, TRAIN_LON_GRID_SIZE
            )
            transition_counts[(g1, g2)] += 1

    return transition_counts, lat_min, lon_min


transition_counts, train_lat_min, train_lon_min = build_interaction_counts(train_raw)

print("interaction transitions:", len(transition_counts))

interaction transitions: 5159


In [27]:
# 10. Multi-Graph 생성

DISTANCE_GRAPH_SIGMA_KM = 0.5


def create_distance_graph(raw_seq, sigma_km=DISTANCE_GRAPH_SIGMA_KM):
    coords = raw_seq[:, :2]

    lat_km = coords[:, 0] * 111.0
    lon_km = coords[:, 1] * 111.0 * np.cos(np.radians(BASE_LAT_FOR_LON_SCALE))
    xy = np.stack([lat_km, lon_km], axis=1)

    diff = xy[:, None, :] - xy[None, :, :]
    dist_km = np.sqrt(np.sum(diff ** 2, axis=-1))

    A = np.exp(-dist_km / sigma_km)
    return A.astype(np.float32)


def create_interaction_graph(raw_seq):
    N = len(raw_seq)
    A = np.zeros((N, N), dtype=np.float32)

    nodes = [
        latlon_to_grid(
            row[0], row[1],
            train_lat_min, train_lon_min,
            TRAIN_LAT_GRID_SIZE, TRAIN_LON_GRID_SIZE
        )
        for row in raw_seq
    ]

    for i in range(N):
        for j in range(N):
            A[i, j] = transition_counts.get((nodes[i], nodes[j]), 0)

    if A.max() > 0:
        A = A / (A.max() + 1e-6)

    return A.astype(np.float32)


def create_multi_graph(raw_seq):
    return np.stack([
        create_distance_graph(raw_seq),
        create_interaction_graph(raw_seq)
    ], axis=0).astype(np.float32)

In [28]:
# 11. Memory-safe sequence index
# 30초 연속 sequence만 사용

SEQ_LEN = 20
TARGET_GAP = np.timedelta64(30, "s")


def build_sequence_items(raw_df, seq_len=20):
    items = []

    for mmsi, group in raw_df.groupby("MMSI"):
        group = group.sort_values("timestamp").reset_index()
        timestamps = group["timestamp"].values.astype("datetime64[ns]")

        if len(group) <= seq_len:
            continue

        for start_idx in range(0, len(group) - seq_len):
            time_window = timestamps[start_idx:start_idx + seq_len + 1]
            time_diffs = np.diff(time_window).astype("timedelta64[s]")

            if not np.all(time_diffs == TARGET_GAP):
                continue

            items.append((mmsi, start_idx))

    return items


train_items = build_sequence_items(train_raw, SEQ_LEN)
val_items = build_sequence_items(val_raw, SEQ_LEN)
test_items = build_sequence_items(test_raw, SEQ_LEN)

print("train sequences:", len(train_items))
print("val sequences:", len(val_items))
print("test sequences:", len(test_items))

train sequences: 164720
val sequences: 545952
test sequences: 175423


In [29]:
# 12. Dataset / DataLoader

class AISDataset(Dataset):
    def __init__(self, raw_df, norm_df, items, seq_len=20):
        self.raw_groups = {
            mmsi: group.sort_values("timestamp").reset_index(drop=True)
            for mmsi, group in raw_df.groupby("MMSI")
        }
        self.norm_groups = {
            mmsi: group.sort_values("timestamp").reset_index(drop=True)
            for mmsi, group in norm_df.groupby("MMSI")
        }
        self.items = items
        self.seq_len = seq_len

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        mmsi, start_idx = self.items[idx]

        raw_group = self.raw_groups[mmsi]
        norm_group = self.norm_groups[mmsi]

        raw_seq_df = raw_group.iloc[start_idx:start_idx + self.seq_len]
        norm_seq_df = norm_group.iloc[start_idx:start_idx + self.seq_len]

        current_pos = raw_group.iloc[start_idx + self.seq_len - 1][["lat", "lon"]].values.astype(np.float32)
        next_pos = raw_group.iloc[start_idx + self.seq_len][["lat", "lon"]].values.astype(np.float32)
        raw_delta = next_pos - current_pos

        target = y_scaler.transform(
            pd.DataFrame([raw_delta], columns=target_cols)
        )[0].astype(np.float32)

        raw_seq = raw_seq_df[feature_cols].values.astype(np.float32)
        norm_seq = norm_seq_df[feature_cols].values.astype(np.float32)
        graph = create_multi_graph(raw_seq)

        return (
            torch.tensor(norm_seq, dtype=torch.float32),
            torch.tensor(graph, dtype=torch.float32),
            torch.tensor(target, dtype=torch.float32)
        )


BATCH_SIZE = 64

train_dataset = AISDataset(train_raw, train_norm, train_items, SEQ_LEN)
val_dataset = AISDataset(val_raw, val_norm, val_items, SEQ_LEN)
test_dataset = AISDataset(test_raw, test_norm, test_items, SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

for X, graphs, y in train_loader:
    print("X:", X.shape)
    print("graphs:", graphs.shape)
    print("y:", y.shape)
    break

X: torch.Size([64, 20, 7])
graphs: torch.Size([64, 2, 20, 20])
y: torch.Size([64, 2])


In [30]:
# 13. STMGCN 모델

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.1):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, X, A):
        B, T, _ = X.shape
        I = torch.eye(T, device=X.device, dtype=X.dtype).unsqueeze(0).expand(B, T, T)

        A_hat = A.to(dtype=X.dtype) + I
        degree = torch.sum(A_hat, dim=-1).clamp_min(1e-6)
        degree_inv_sqrt = torch.pow(degree, -0.5)
        A_norm = degree_inv_sqrt.unsqueeze(-1) * A_hat * degree_inv_sqrt.unsqueeze(-2)

        H = A_norm @ self.linear(X)
        H = F.relu(H)
        H = self.dropout(H)

        return self.norm(H)


class MultiGraphAttentionFusion(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.score_layer = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, H_list):
        scores = [self.score_layer(H).mean(dim=1) for H in H_list]
        scores = torch.cat(scores, dim=1)
        weights = F.softmax(scores, dim=1)

        fused = torch.zeros_like(H_list[0])
        for i, H in enumerate(H_list):
            fused = fused + weights[:, i].view(-1, 1, 1) * H

        return fused, weights


class TemporalConvBlock(nn.Module):
    def __init__(self, hidden_dim, kernel_size=3, dropout=0.1):
        super().__init__()
        padding = kernel_size // 2

        self.filter_conv = nn.Conv1d(hidden_dim, hidden_dim, kernel_size, padding=padding)
        self.gate_conv = nn.Conv1d(hidden_dim, hidden_dim, kernel_size, padding=padding)
        self.proj = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=1)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        residual = x

        x = x.transpose(1, 2)
        z = torch.tanh(self.filter_conv(x))
        g = torch.sigmoid(self.gate_conv(x))
        x = self.proj(z * g)
        x = self.dropout(x)
        x = x.transpose(1, 2)

        return self.norm(x + residual)


class STMGCN(nn.Module):
    def __init__(
        self,
        input_dim=7,
        hidden_dim=64,
        num_graphs=2,
        output_dim=2,
        num_temporal_blocks=2,
        dropout=0.1
    ):
        super().__init__()

        self.num_graphs = num_graphs

        self.input_projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim)
        )

        self.gcn_layers = nn.ModuleList([
            GCNLayer(hidden_dim, hidden_dim, dropout=dropout)
            for _ in range(num_graphs)
        ])

        self.graph_fusion = MultiGraphAttentionFusion(hidden_dim)

        self.temporal_blocks = nn.ModuleList([
            TemporalConvBlock(hidden_dim, dropout=dropout)
            for _ in range(num_temporal_blocks)
        ])

        self.output_layer = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim)
        )

    def forward(self, X, graphs):
        H0 = self.input_projection(X)

        H_list = []
        for i in range(self.num_graphs):
            H_list.append(self.gcn_layers[i](H0, graphs[:, i]))

        H, _ = self.graph_fusion(H_list)

        for block in self.temporal_blocks:
            H = block(H)

        return self.output_layer(H[:, -1, :])

In [31]:
# 14. 학습 준비

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

model = STMGCN(
    input_dim=len(feature_cols),
    hidden_dim=64,
    num_graphs=2,
    output_dim=2,
    num_temporal_blocks=2,
    dropout=0.1
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
criterion = nn.MSELoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=10,
    min_lr=1e-5
)

device: cuda


In [ ]:
# 15. 학습

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0

    for X, graphs, y in loader:
        X = X.to(device)
        graphs = graphs.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        pred = model(X, graphs)
        loss = criterion(pred, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate_loss(model, loader):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for X, graphs, y in loader:
            X = X.to(device)
            graphs = graphs.to(device)
            y = y.to(device)

            pred = model(X, graphs)
            loss = criterion(pred, y)
            total_loss += loss.item()

    return total_loss / len(loader)


EPOCHS = 300
PATIENCE = 30

best_val_loss = float("inf")
best_model_state = None
patience_counter = 0

train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader)
    val_loss = evaluate_loss(model, val_loader)

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch + 1}/{EPOCHS}] Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("Early stopping")
        break

if best_model_state is not None:
    model.load_state_dict(best_model_state)

print("Best Val Loss:", best_val_loss)

Epoch [1/300] Train Loss: 0.003153 | Val Loss: 0.000230
Epoch [2/300] Train Loss: 0.000552 | Val Loss: 0.000256
Epoch [3/300] Train Loss: 0.000404 | Val Loss: 0.000189
Epoch [4/300] Train Loss: 0.000335 | Val Loss: 0.000166
Epoch [5/300] Train Loss: 0.000307 | Val Loss: 0.000174
Epoch [6/300] Train Loss: 0.000297 | Val Loss: 0.000186
Epoch [7/300] Train Loss: 0.000291 | Val Loss: 0.000189
Epoch [8/300] Train Loss: 0.000289 | Val Loss: 0.000162
Epoch [9/300] Train Loss: 0.000284 | Val Loss: 0.000170
Epoch [10/300] Train Loss: 0.000281 | Val Loss: 0.000168
Epoch [11/300] Train Loss: 0.000278 | Val Loss: 0.000165
Epoch [12/300] Train Loss: 0.000281 | Val Loss: 0.000213
Epoch [13/300] Train Loss: 0.000276 | Val Loss: 0.000160
Epoch [14/300] Train Loss: 0.000277 | Val Loss: 0.000153
Epoch [15/300] Train Loss: 0.000277 | Val Loss: 0.000166
Epoch [16/300] Train Loss: 0.000277 | Val Loss: 0.000156
Epoch [17/300] Train Loss: 0.000278 | Val Loss: 0.000224


In [ ]:
# 16. 학습 곡선

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training / Validation Loss")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# 17. 30초 단위 recursive 예측 함수

FORECAST_SECONDS_LIST = [30, 60, 300, 600, 900]
MAX_FORECAST_SECONDS = max(FORECAST_SECONDS_LIST)
TOLERANCE_SECONDS = 15


def haversine_km(lat1, lon1, lat2, lon2):
    return haversine_km_np(lat1, lon1, lat2, lon2)


def estimate_sog_cog_from_points(prev_lat, prev_lon, new_lat, new_lon, seconds=30):
    dist_km = haversine_km(prev_lat, prev_lon, new_lat, new_lon)
    sog_knots = (dist_km / 1.852) / (seconds / 3600)

    dlon = np.radians(new_lon - prev_lon)
    lat1 = np.radians(prev_lat)
    lat2 = np.radians(new_lat)

    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)

    cog = (np.degrees(np.arctan2(y, x)) + 360) % 360
    return sog_knots, cog


def predict_next_position(model, raw_seq, norm_seq):
    graph = create_multi_graph(raw_seq)

    X = torch.tensor(norm_seq, dtype=torch.float32).unsqueeze(0).to(device)
    graphs = torch.tensor(graph, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        pred_delta_norm = model(X, graphs).cpu().numpy()

    pred_delta = y_scaler.inverse_transform(pred_delta_norm)[0]
    pred_latlon = raw_seq[-1, :2] + pred_delta

    return pred_delta, pred_latlon


def recursive_forecast_path(model, raw_seq, norm_seq, forecast_seconds=900):
    steps = int(forecast_seconds / STEP_SECONDS)

    raw_seq = raw_seq.copy()
    norm_seq = norm_seq.copy()
    predicted_path = []

    for step in range(1, steps + 1):
        pred_delta, pred_latlon = predict_next_position(model, raw_seq, norm_seq)

        prev_raw = raw_seq[-1].copy()
        new_raw = prev_raw.copy()

        new_raw[0] = pred_latlon[0]
        new_raw[1] = pred_latlon[1]

        est_sog, est_cog = estimate_sog_cog_from_points(
            prev_raw[0], prev_raw[1],
            pred_latlon[0], pred_latlon[1],
            seconds=STEP_SECONDS
        )

        new_raw[2] = est_sog
        new_raw[3] = np.sin(np.radians(est_cog))
        new_raw[4] = np.cos(np.radians(est_cog))
        new_raw[5] = np.sin(np.radians(est_cog))
        new_raw[6] = np.cos(np.radians(est_cog))

        new_norm = x_scaler.transform(pd.DataFrame([new_raw], columns=feature_cols))[0]

        raw_seq = np.vstack([raw_seq[1:], new_raw])
        norm_seq = np.vstack([norm_seq[1:], new_norm])

        predicted_path.append({
            "step": step,
            "forecast_seconds": step * STEP_SECONDS,
            "forecast_minutes": step * STEP_SECONDS / 60,
            "lat": pred_latlon[0],
            "lon": pred_latlon[1]
        })

    return pd.DataFrame(predicted_path)


def get_actual_position_near_time(df, mmsi, target_time, tolerance_seconds=15):
    tolerance = pd.Timedelta(seconds=tolerance_seconds)

    vessel_df = df[
        (df["MMSI"] == mmsi) &
        (df["timestamp"] >= target_time - tolerance) &
        (df["timestamp"] <= target_time + tolerance)
    ].copy()

    if len(vessel_df) == 0:
        return None

    vessel_df["time_diff"] = (vessel_df["timestamp"] - target_time).abs()
    row = vessel_df.sort_values("time_diff").iloc[0]

    return {
        "timestamp": row["timestamp"],
        "lat": row["lat"],
        "lon": row["lon"]
    }


def get_actual_path_for_horizons(df, mmsi, base_time, forecast_seconds_list):
    rows = []

    for seconds in forecast_seconds_list:
        actual = get_actual_position_near_time(
            df,
            mmsi,
            base_time + pd.Timedelta(seconds=seconds),
            tolerance_seconds=TOLERANCE_SECONDS
        )

        if actual is None:
            continue

        rows.append({
            "forecast_seconds": seconds,
            "forecast_minutes": seconds / 60,
            "timestamp": actual["timestamp"],
            "lat": actual["lat"],
            "lon": actual["lon"]
        })

    return pd.DataFrame(rows)

In [ ]:
# 18. 8월 test 전체 선박 / 전체 가능한 시점 예측

test_trajectory_records = []
test_trajectory_paths = []

total_candidates = 0
used_candidates = 0

model.eval()

for mmsi_idx, (mmsi, raw_group) in enumerate(test_raw.groupby("MMSI"), start=1):
    raw_group = raw_group.sort_values("timestamp").reset_index(drop=True)
    norm_group = test_norm[test_norm["MMSI"] == mmsi].sort_values("timestamp").reset_index(drop=True)

    if len(raw_group) < SEQ_LEN + 1:
        continue

    raw_data = raw_group[feature_cols].values.astype(np.float32)
    norm_data = norm_group[feature_cols].values.astype(np.float32)
    timestamps = raw_group["timestamp"].values.astype("datetime64[ns]")

    for start_idx in range(0, len(raw_group) - SEQ_LEN):
        total_candidates += 1

        time_window = timestamps[start_idx:start_idx + SEQ_LEN]
        time_diffs = np.diff(time_window).astype("timedelta64[s]")

        if not np.all(time_diffs == TARGET_GAP):
            continue

        base_time = pd.Timestamp(timestamps[start_idx + SEQ_LEN - 1])

        raw_seq = raw_data[start_idx:start_idx + SEQ_LEN]
        norm_seq = norm_data[start_idx:start_idx + SEQ_LEN]

        pred_path = recursive_forecast_path(
            model,
            raw_seq,
            norm_seq,
            forecast_seconds=MAX_FORECAST_SECONDS
        )

        actual_path = get_actual_path_for_horizons(
            test_raw,
            mmsi,
            base_time,
            FORECAST_SECONDS_LIST
        )

        if len(actual_path) == 0:
            continue

        pred_horizon = pred_path[
            pred_path["forecast_seconds"].isin(FORECAST_SECONDS_LIST)
        ].copy()

        merged = pred_horizon.merge(
            actual_path,
            on=["forecast_seconds", "forecast_minutes"],
            suffixes=("_pred", "_actual")
        )

        if len(merged) == 0:
            continue

        used_candidates += 1

        for _, row in merged.iterrows():
            error_km = haversine_km(
                row["lat_actual"], row["lon_actual"],
                row["lat_pred"], row["lon_pred"]
            )

            test_trajectory_records.append({
                "base_time": base_time,
                "MMSI": mmsi,
                "start_sog": raw_seq[-1][2],
                "forecast_seconds": row["forecast_seconds"],
                "forecast_minutes": row["forecast_minutes"],
                "pred_lat": row["lat_pred"],
                "pred_lon": row["lon_pred"],
                "actual_lat": row["lat_actual"],
                "actual_lon": row["lon_actual"],
                "error_km": error_km,
                "error_m": error_km * 1000,
                "error_nm": error_km / 1.852
            })

        test_trajectory_paths.append({
            "base_time": base_time,
            "MMSI": mmsi,
            "start_lat": raw_seq[-1][0],
            "start_lon": raw_seq[-1][1],
            "start_sog": raw_seq[-1][2],
            "raw_seq": raw_seq,
            "pred_path": pred_horizon.reset_index(drop=True),
            "actual_path": actual_path.reset_index(drop=True)
        })

    if mmsi_idx % 20 == 0:
        print(f"processed vessels: {mmsi_idx}, total: {total_candidates}, used: {used_candidates}")

test_trajectory_results_df = pd.DataFrame(test_trajectory_records)

print("total candidates:", total_candidates)
print("used candidates:", used_candidates)
print("result rows:", len(test_trajectory_results_df))
print("paths:", len(test_trajectory_paths))

display(test_trajectory_results_df.head())

In [ ]:
# 19. 성능평가

trajectory_summary_df = (
    test_trajectory_results_df
    .groupby("forecast_seconds")
    .agg(
        forecast_minutes=("forecast_minutes", "first"),
        valid_count=("error_km", "count"),
        vessel_count=("MMSI", "nunique"),
        mean_error_km=("error_km", "mean"),
        median_error_km=("error_km", "median"),
        rmse_km=("error_km", lambda x: np.sqrt(np.mean(np.square(x)))),
        p90_error_km=("error_km", lambda x: np.percentile(x, 90)),
        max_error_km=("error_km", "max"),
        mean_error_m=("error_m", "mean"),
        median_error_m=("error_m", "median")
    )
    .reset_index()
)

display(trajectory_summary_df)

plt.figure(figsize=(10, 5))
plt.plot(trajectory_summary_df["forecast_minutes"], trajectory_summary_df["mean_error_m"], marker="o", label="Mean")
plt.plot(trajectory_summary_df["forecast_minutes"], trajectory_summary_df["median_error_m"], marker="s", label="Median")
plt.xlabel("Forecast Horizon (minutes)")
plt.ylabel("Error (m)")
plt.title("30-second STMGCN Prediction Error")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# 20. 예측 상위 5 / 하위 5 그래프
# 기준: 30초, 1분, 5분, 10분, 15분 평균 오차

SELECTION_SECONDS = [30, 60, 300, 600, 900]
TOP_N = 5
BOTTOM_N = 5


def build_plot_dataframe(results_df, paths):
    result_df = results_df.copy().rename(columns={
        "MMSI": "mmsi",
        "actual_lat": "true_lat",
        "actual_lon": "true_lon"
    })

    start_rows = []

    for sample_index, item in enumerate(paths):
        start_rows.append({
            "sample_index": sample_index,
            "mmsi": item["MMSI"],
            "base_time": item["base_time"],
            "last_lat": item["start_lat"],
            "last_lon": item["start_lon"],
            "start_sog": item["start_sog"]
        })

    start_df = pd.DataFrame(start_rows)

    result_df = result_df.merge(
        start_df,
        on=["mmsi", "base_time"],
        how="inner"
    )

    return result_df.dropna()


def select_top_bottom_samples(plot_df):
    horizon_df = plot_df[plot_df["forecast_seconds"].isin(SELECTION_SECONDS)].copy()

    sample_score_df = (
        horizon_df
        .groupby("sample_index")
        .agg(
            mmsi=("mmsi", "first"),
            base_time=("base_time", "first"),
            start_sog=("start_sog", "first"),
            mean_error_m=("error_m", "mean"),
            median_error_m=("error_m", "median"),
            max_error_m=("error_m", "max"),
            count=("error_m", "count")
        )
        .reset_index()
    )

    sample_score_df = sample_score_df[
        sample_score_df["count"] >= len(SELECTION_SECONDS)
    ].copy()

    top_samples = sample_score_df.sort_values("mean_error_m", ascending=True).head(TOP_N)
    bottom_samples = sample_score_df.sort_values("mean_error_m", ascending=False).head(BOTTOM_N)

    return sample_score_df, top_samples, bottom_samples


def plot_latlon_pred_actual(plot_df, selected_samples, title, draw_error_lines=False):
    selected_ids = selected_samples["sample_index"].tolist()

    data = plot_df[
        plot_df["sample_index"].isin(selected_ids) &
        plot_df["forecast_seconds"].isin(SELECTION_SECONDS)
    ].copy()

    plt.figure(figsize=(11, 9))
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(selected_ids), 1)))

    for i, sample_index in enumerate(selected_ids):
        group = data[data["sample_index"] == sample_index].sort_values("forecast_seconds")

        if len(group) == 0:
            continue

        color = colors[i]
        mmsi = int(group.iloc[0]["mmsi"])

        plt.scatter(
            group.iloc[0]["last_lon"],
            group.iloc[0]["last_lat"],
            color=color,
            marker="o",
            s=70,
            edgecolor="black"
        )

        plt.plot(
            group["true_lon"],
            group["true_lat"],
            color=color,
            linestyle="-",
            marker="x",
            linewidth=1.5,
            label=f"Actual {i}: {mmsi}"
        )

        plt.plot(
            group["pred_lon"],
            group["pred_lat"],
            color=color,
            linestyle="--",
            marker="*",
            linewidth=1.5,
            label=f"Pred {i}: {mmsi}"
        )

        for _, row in group.iterrows():
            label = f"{int(row['forecast_seconds'])}s" if row["forecast_seconds"] < 60 else f"{row['forecast_minutes']:.0f}min"
            plt.text(row["pred_lon"], row["pred_lat"], label, fontsize=7, color=color)

        if draw_error_lines:
            for _, row in group.iterrows():
                plt.plot(
                    [row["true_lon"], row["pred_lon"]],
                    [row["true_lat"], row["pred_lat"]],
                    color=color,
                    alpha=0.35,
                    linewidth=0.9
                )

    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title(title)
    plt.grid(True)
    plt.legend(fontsize=8, ncol=2)

    ax = plt.gca()
    ax.xaxis.set_major_formatter(ScalarFormatter(useOffset=False))
    ax.yaxis.set_major_formatter(ScalarFormatter(useOffset=False))
    ax.ticklabel_format(style="plain", axis="both")

    plt.show()


plot_df = build_plot_dataframe(test_trajectory_results_df, test_trajectory_paths)
score_df, top5_df, bottom5_df = select_top_bottom_samples(plot_df)

print("Top 5 samples")
display(top5_df)

print("Bottom 5 samples")
display(bottom5_df)

plot_latlon_pred_actual(plot_df, top5_df, "Top 5: Predicted vs Actual", draw_error_lines=False)
plot_latlon_pred_actual(plot_df, top5_df, "Top 5: Error Lines", draw_error_lines=True)

plot_latlon_pred_actual(plot_df, bottom5_df, "Bottom 5: Predicted vs Actual", draw_error_lines=False)
plot_latlon_pred_actual(plot_df, bottom5_df, "Bottom 5: Error Lines", draw_error_lines=True)